# Project RxHARM — Notebook 01: HVI & HRI Computation
**Prescriptive Optimization of Heat Associated Risk Mitigation**

This notebook computes the **Heat Vulnerability Index (HVI)** and **Heat Risk Index (HRI)**
at 100-meter resolution for any city or region using freely available satellite data.

**Outputs:** GeoTIFF rasters of HVI, HRI, and all 14 sub-indicators saved to Google Drive.

📄 Paper DOI: `https://doi.org/PLACEHOLDER`  
💻 GitHub: `https://github.com/YOUR_USERNAME/rxharm`

> **Before you begin:** Run cells in order. Each cell must complete before running the next.
> Cells 1–3 are setup; Cell 5 is the only cell you need to edit.

In [ ]:
# ── Cell 2: Installation ────────────────────────────────────────────────────
# Install the RxHARM package directly from GitHub.
# --force-reinstall ensures the latest commit is always used.

!pip install --force-reinstall --no-cache-dir -q git+https://github.com/N-A-F-I-Z/RxHARM.git

# Verify installation
import rxharm
print(f'RxHARM version: {rxharm.__version__} installed successfully')


In [ ]:
# ── Cell 3: Authentication ──────────────────────────────────────────────────
# Authenticate Google Earth Engine.
# You will be prompted to log in with your Google account.

import ee
ee.Authenticate()
ee.Initialize(project='YOUR_GEE_PROJECT_ID')  # <-- replace with your GEE project ID

# Mount Google Drive for saving outputs
from google.colab import drive
drive.mount('/content/drive')

print('Authentication complete.')

## USER INPUTS — Edit the values in the cell below

This is the **only cell you need to edit.** Change the values to match your study area.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# USER INPUTS — Only edit values in this cell
# ══════════════════════════════════════════════════════════════════════════════

# Choose ONE of the following three AOI methods. Set the other two to None/False.
# 1. City name (e.g., 'Ahmedabad, India')
LOCATION = 'Ahmedabad, India'

# 2. Path to a local shapefile (e.g., '/content/my_city.shp')
SHAPEFILE_PATH = None

# 3. Draw AOI interactively using geemap (set to True to draw on map)
DRAW_AOI = False

# Analysis year (2016-2024 supported)
YEAR = 2023

# Weighting method for indicator aggregation ('equal', 'pca', 'ahp')
WEIGHTING_METHOD = 'equal'

# Google Drive folder for saving outputs
DRIVE_OUTPUT_FOLDER = 'RxHARM_outputs'
EXPORT_TO_DRIVE = True

print(f'Configuration loaded for Year: {YEAR} | Method: {WEIGHTING_METHOD}')


In [ ]:
# ── AOI Processing: Step 1 (Map) ─────────────────────────────────────────────
import ee
import geemap
from rxharm.aoi.handler import AOIHandler

if DRAW_AOI:
    print("Please draw a polygon on the map below for your AOI. Then run the NEXT cell.")
    Map = geemap.Map()
    display(Map)
else:
    print("Using pre-defined Location or Shapefile. Proceed to next cell.")


In [ ]:
# ── AOI Processing: Step 2 (Extraction) ──────────────────────────────────────
if DRAW_AOI:
    if not Map.draw_features:
        raise ValueError("No polygon drawn! Please draw a polygon on the map above.")
    ee_geom = ee.FeatureCollection(Map.draw_features).geometry()
    aoi = AOIHandler(ee_geom, YEAR)
elif SHAPEFILE_PATH:
    import geopandas as gpd
    print(f"Loading shapefile from {SHAPEFILE_PATH}...")
    gdf = gpd.read_file(SHAPEFILE_PATH)
    geom = list(gdf.geometry)[0]
    coords = list(geom.exterior.coords)
    ee_geom = ee.Geometry.Polygon(coords)
    aoi = AOIHandler(ee_geom, YEAR)
else:
    print(f'Processing AOI for {LOCATION}...')
    aoi = AOIHandler(LOCATION, YEAR)

aoi.validate()
aoi.display_summary()


In [ ]:
# ── Parallel Execution: GEE, WorldPop, VIIRS ───────────────────────────────
import concurrent.futures
import time
import os
import numpy as np
from rxharm.seasonal.detector import SeasonalDetector
from rxharm.fetch import fetch_all_indicators
from rxharm.fetch.worldpop_fetcher import WorldPopFetcher
from rxharm.fetch.viirs_downscaler import VIIRSDownscaler
import geemap
import ee

print('Detecting hottest period from ERA5 climatology...')
detector = SeasonalDetector(aoi.centroid_ll[0], aoi.centroid_ll[1], YEAR)
hottest_months = detector.detect(use_cache=True)
print(f'Hottest months: {hottest_months}')

# ── Task 1: GEE Export (sequential — submits async Drive export) ─────────
print('[GEE Task] Submitting indicator fetch and export...')
try:
    gee_res = fetch_all_indicators(
        aoi_handler=aoi,
        seasonal_detector=detector,
        export_to_drive=EXPORT_TO_DRIVE,
        drive_folder=DRIVE_OUTPUT_FOLDER,
        show_progress=True
    )
    if gee_res.get('export_task'):
        print(f'[GEE Task] Export started. Monitor: https://code.earthengine.google.com/tasks')
except Exception as e:
    print(f'[GEE Task] Failed: {e}')
    gee_res = None

# ── Tasks 2 & 3: WorldPop + VIIRS (parallel local downloads) ─────────────
def task_worldpop():
    print('[WorldPop Task] Downloading 100m population bands locally...')
    iso3 = getattr(aoi, 'iso3', 'BGD')
    wp_fetcher = WorldPopFetcher(iso3, YEAR)
    geom = aoi.to_ee_geometry()
    bounds_info = ee.Geometry(geom).bounds().getInfo()['coordinates'][0]
    lons = [c[0] for c in bounds_info]
    lats = [c[1] for c in bounds_info]
    bbox = (min(lons), min(lats), max(lons), max(lats))
    bands = wp_fetcher.download_required_bands(bounds=bbox)
    print('[WorldPop Task] Done.')
    return bands

def task_viirs():
    print('[VIIRS Task] Fetching raw bands for GPU downscaling...')
    from rxharm.fetch.adaptive_capacity import AdaptiveCapacityFetcher
    ac = AdaptiveCapacityFetcher(aoi.to_ee_geometry(), YEAR, hottest_months)
    viirs_raw = ac.get_viirs_dnb_raw()
    viirs_path = 'viirs_raw.tif'
    geemap.ee_export_image(
        viirs_raw, filename=viirs_path, scale=450,
        region=aoi.to_ee_geometry(), file_per_band=False
    )
    print('[VIIRS Task] Raw VIIRS fetched.')
    return viirs_path

print('\n\U0001f680 Launching parallel local downloads (WorldPop & VIIRS)...')
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
    f_wp = executor.submit(task_worldpop)
    f_viirs = executor.submit(task_viirs)
    wp_res = f_wp.result()
    viirs_path = f_viirs.result()

print('\n\u2705 Data fetch complete!')

# Debug: show what keys WorldPop returned
print(f'  WorldPop keys: {list(wp_res.keys()) if wp_res else "None"}')

# ── Compute elderly_frac and child_frac from WorldPop bands ──────────────
if wp_res and 'total' in wp_res:
    total_pop = np.nan_to_num(wp_res['total'], nan=0.0)
    total_pop_safe = np.where(total_pop > 0, total_pop, 1.0)  # avoid /0

    # Sum child bands — handle both key formats: child_0/child_1 AND child_00/child_01
    child_sum = np.zeros_like(total_pop)
    for key in ['child_0', 'child_1', 'child_00', 'child_01']:
        if key in wp_res and wp_res[key] is not None:
            child_sum += np.nan_to_num(wp_res[key], nan=0.0)
    child_frac = child_sum / total_pop_safe

    # Sum elderly bands (age 60, 65, 70, 75, 80, 85, 90)
    elderly_sum = np.zeros_like(total_pop)
    for key in ['elderly_60', 'elderly_65', 'elderly_70', 'elderly_75',
                'elderly_80', 'elderly_85', 'elderly_90']:
        if key in wp_res and wp_res[key] is not None:
            elderly_sum += np.nan_to_num(wp_res[key], nan=0.0)
    elderly_frac = elderly_sum / total_pop_safe

    print(f'  Child fraction : min={child_frac.min():.4f}  max={child_frac.max():.4f}  mean={child_frac.mean():.4f}')
    print(f'  Elderly fraction: min={elderly_frac.min():.4f}  max={elderly_frac.max():.4f}  mean={elderly_frac.mean():.4f}')
    print(f'  Total population: shape={total_pop.shape}  sum={total_pop.sum():.0f}')

    # Save as GeoTIFFs for the downscaler and later merge
    import rasterio
    from rasterio.transform import from_bounds
    geom = aoi.to_ee_geometry()
    bounds_info = ee.Geometry(geom).bounds().getInfo()['coordinates'][0]
    lons = [c[0] for c in bounds_info]
    lats = [c[1] for c in bounds_info]
    h, w = total_pop.shape
    wp_profile = {
        'driver': 'GTiff', 'dtype': 'float32', 'count': 1,
        'height': h, 'width': w, 'crs': 'EPSG:4326',
        'transform': from_bounds(min(lons), min(lats), max(lons), max(lats), w, h),
    }
    for name, arr in [('worldpop_total.tif', total_pop),
                      ('worldpop_child_frac.tif', child_frac),
                      ('worldpop_elderly_frac.tif', elderly_frac)]:
        with rasterio.open(name, 'w', **wp_profile) as dst:
            dst.write(arr.astype('float32'), 1)
    print('  Saved: worldpop_total.tif, worldpop_child_frac.tif, worldpop_elderly_frac.tif')
else:
    print('\u26a0 WorldPop download returned no data.')

# ── VIIRS Downscaling ────────────────────────────────────────────────────
if wp_res and 'total' in wp_res:
    print('\nRunning VIIRS RFATPK downscaling...')
    downscaler = VIIRSDownscaler(backend='gpu', zoom=4.5)
    try:
        viirs_100m, fine_profile = downscaler.downscale(
            viirs_path=viirs_path,
            pop_path='worldpop_total.tif',
        )
        print(f'\u2705 VIIRS downscaling complete: {viirs_100m.shape}')
    except Exception as e:
        print(f'\u26a0 VIIRS downscaling failed: {e}')
        import traceback; traceback.print_exc()
else:
    print('Skipping VIIRS downscaling (WorldPop data missing).')


In [ ]:
# ── Step 4: Merge Data & Compute HVI & HRI ─────────────────────────────────
# Wait for the GEE Task to complete before running this!
# You can check your Google Drive for a file named something like:
# 'RxHARM_indicators_2025_...tif'

import glob
import os
import rasterio
from rxharm.fetch import load_existing_export, merge_worldpop_local
from rxharm.index.hvi import HVIEngine
from rxharm.index.hri import HRIEngine

# Find the newest downloaded TIFF in your Drive folder
drive_path = f'/content/drive/MyDrive/{DRIVE_OUTPUT_FOLDER}/*.tif'
tiffs = glob.glob(drive_path)
if not tiffs:
    print(f'⚠ No TIFF files found in {drive_path}. Is the GEE export finished?')
else:
    latest_tiff = max(tiffs, key=os.path.getmtime)
    print(f'Loading latest GEE export: {latest_tiff}')
    
    # 1. Load GEE data
    data = load_existing_export(latest_tiff)
    
    # 2. Merge local WorldPop data
    data = merge_worldpop_local(data)
    
    # 3. Inject downscaled VIIRS data
    if 'viirs_100m' in globals() and 'viirs' in data['arrays']:
        from PIL import Image as _PIL
        gee_shape = data['arrays']['viirs'].shape
        if viirs_100m.shape != gee_shape:
            img = _PIL.fromarray(viirs_100m.astype(np.float32))
            viirs_resized = np.array(img.resize((gee_shape[1], gee_shape[0]), _PIL.BILINEAR))
            data['arrays']['viirs'] = viirs_resized
        else:
            data['arrays']['viirs'] = viirs_100m
        print('  ✅ VIIRS 100m downscaled array injected into data dictionary.')
        
    # 4. Run HVI Engine
    print('\nRunning HVI Engine...')
    hvi_engine = HVIEngine(weighting_method=WEIGHTING_METHOD)
    hvi_results = hvi_engine.compute_all(data['arrays'])
    
    # 5. Run HRI Engine
    print('Running HRI Engine...')
    iso3 = getattr(aoi, 'iso3', 'IND')
    # Try to get climate zone, default to 'C' (temperate/mesothermal) if not available
    climate_zone = getattr(aoi, 'climate_zone', 'C') 
    if not isinstance(climate_zone, str) or climate_zone not in ['A', 'B', 'C', 'D', 'E']:
        climate_zone = 'C'
        
    hri_engine = HRIEngine(climate_zone=climate_zone, cdr_baseline=iso3)
    
    # Make sure MMT is computed first
    if 'lst' in hvi_results['indicator_normalized']:
        hri_engine.compute_mmt(hvi_results['indicator_normalized']['lst'])
    else:
        hri_engine.compute_mmt(data['arrays'].get('lst', np.zeros_like(hvi_results['HVI'])))
        
    hri_results = hri_engine.compute_all(hvi_results)
    
    # Combine all results
    final_results = {**hvi_results, **hri_results}
    
    # 6. Save Final Outputs to Drive
    output_tiff = latest_tiff.replace('indicators', 'results_HVI_HRI')
    print(f'\nExporting final results to: {output_tiff}')
    
    # Extract profile from original TIFF
    with rasterio.open(latest_tiff) as src:
        profile = src.profile
    
    bands_to_export = ['H_s', 'E', 'S', 'AC', 'HVI', 'HRI', 'AD_baseline', 'AF']
    profile.update(count=len(bands_to_export), dtype='float32')
    
    with rasterio.open(output_tiff, 'w', **profile) as dst:
        for i, band_name in enumerate(bands_to_export, 1):
            dst.write(final_results[band_name].astype('float32'), i)
            dst.set_band_description(i, band_name)
            
    print('✅ HVI/HRI Computation and Export Complete!')


In [ ]:
# ── Step 5: Visualize Final Results ────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

if 'final_results' in locals():
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    plots = [
        ('H_s', 'Hazard (H_s)', 'Reds'),
        ('E', 'Exposure (E)', 'Oranges'),
        ('S', 'Sensitivity (S)', 'Purples'),
        ('AC', 'Adaptive Capacity (AC)', 'Greens_r'),
        ('HVI', 'Heat Vulnerability Index (HVI)', 'turbo'),
        ('HRI', 'Heat Risk Index (HRI)', 'magma')
    ]
    
    for i, (key, title, cmap) in enumerate(plots):
        im = axes[i].imshow(final_results[key], cmap=cmap)
        axes[i].set_title(title)
        fig.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)
        axes[i].axis('off')
        
    plt.tight_layout()
    plt.show()
    
    # Print some statistics
    print("\n--- Statistical Summary ---")
    for key in ['HVI', 'HRI', 'AD_baseline']:
        arr = final_results[key]
        print(f"{key:>12}: Min = {np.nanmin(arr):.4f}, Max = {np.nanmax(arr):.4f}, Mean = {np.nanmean(arr):.4f}")
